# Inventory Planning Simulation

This notebook will use the saved 28-day demand forecast to demonstrate inventory planning. Any lead-time or stock assumptions we use are examples, not actual Walmart inventory data.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_dir = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "code" / "notebooks").is_dir()
    ),
    Path.cwd(),
)

forecast_file = (
    project_dir
    / "data"
    / "processed"
    / "forecast_7day_average_28day.csv"
)

forecast_with_metadata = pd.read_csv(forecast_file)

forecast_with_metadata.shape, forecast_with_metadata.head()

## Estimate demand during supplier lead time

We’ll use an assumed 7-day lead time for this example. For each product-store series, we’ll estimate how much demand could occur during those seven days. This is not an order quantity because we don’t have actual stock-on-hand data.

In [ ]:
lead_time_days = 7

inventory_scenario = forecast_with_metadata[
    [
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
        "forecast_28day_total"
    ]
].copy()

inventory_scenario["assumed_lead_time_days"] = lead_time_days

inventory_scenario["expected_demand_during_lead_time"] = (
    inventory_scenario["forecast_28day_total"]
    * lead_time_days
    / 28
)

inventory_scenario.head()

## Estimate safety stock and reorder points

For this example, we assume a 95% service level and a 7-day supplier lead time. We’ll estimate daily demand variability from the last 28 observed days and use it to calculate a safety-stock buffer.

This is a simplified scenario. Real inventory settings would need business data and validation.

In [ ]:
import numpy as np

service_level = 0.95
z_value = 1.645  # Approximate z-score for a 95% service level

sales_file = (
    project_dir
    / "data"
    / "raw"
    / "m5-forecasting-accuracy"
    / "sales_train_validation.csv"
)

last_28_day_columns = [f"d_{day}" for day in range(1886, 1914)]

sales_last_28_days = pd.read_csv(
    sales_file,
    usecols=["id"] + last_28_day_columns
).set_index("id")

daily_sales_std = sales_last_28_days[last_28_day_columns].std(axis=1)

inventory_scenario["daily_sales_std_last_28_days"] = (
    inventory_scenario["id"].map(daily_sales_std)
)

inventory_scenario["safety_stock_units"] = (
    z_value
    * inventory_scenario["daily_sales_std_last_28_days"]
    * np.sqrt(lead_time_days)
)

inventory_scenario["reorder_point_units"] = (
    inventory_scenario["expected_demand_during_lead_time"]
    + inventory_scenario["safety_stock_units"]
)

inventory_scenario[
    [
        "item_id",
        "store_id",
        "expected_demand_during_lead_time",
        "safety_stock_units",
        "reorder_point_units"
    ]
].head()

In [ ]:
inventory_reorder_thresholds = inventory_scenario[
    [
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
        "expected_demand_during_lead_time",
        "safety_stock_units",
        "reorder_point_units"
    ]
].copy()

inventory_reorder_thresholds["reorder_point_units_rounded"] = np.ceil(
    inventory_reorder_thresholds["reorder_point_units"]
).astype(int)

inventory_reorder_thresholds.sort_values(
    "reorder_point_units_rounded",
    ascending=False
).head(10)

In [ ]:
processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

reorder_output_file = (
    processed_dir / "reorder_point_scenario_7day_average.csv"
)

inventory_reorder_thresholds.to_csv(
    reorder_output_file,
    index=False
)

reorder_output_file

## How to interpret these results

Each row gives a scenario reorder point for one product-store series. Under the assumed 7-day lead time and 95% service level, the reorder point is the estimated demand during lead time plus a safety-stock buffer. A store would consider reordering when its inventory position reaches or falls below that threshold.

The M5 dataset does not provide current stock or incoming orders. These results therefore do not say which products Walmart should order today or how many units to order. They are planning thresholds based on stated assumptions.

In [ ]:
store_inventory_summary = (
    inventory_scenario
    .groupby("store_id", as_index=False)
    .agg(
        series_count=("id", "nunique"),
        forecast_28day_units=("forecast_28day_total", "sum"),
        expected_7day_demand=(
            "expected_demand_during_lead_time",
            "sum"
        ),
        total_reorder_point_units=("reorder_point_units", "sum")
    )
    .sort_values("forecast_28day_units", ascending=False)
    .reset_index(drop=True)
)

store_inventory_summary

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.bar(
    store_inventory_summary["store_id"],
    store_inventory_summary["forecast_28day_units"]
)
plt.title("Forecast Demand by Store: Next 28 Days")
plt.xlabel("Store")
plt.ylabel("Forecast units")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Store-level forecast interpretation

This chart adds together the 28-day forecasts for all product–store series in each store. The forecasts use the 7-day average baseline.

CA_3 has the highest forecast demand in this chart, while CA_4 has the lowest. These totals describe predicted sales units; they do not show how much stock a store currently has or how much it should order.

In [ ]:
inventory_reorder_thresholds.head(10)

In [ ]:
inventory_reorder_thresholds.sort_values(
    "reorder_point_units_rounded",
    ascending=False
).head(10)

In [ ]:
store_inventory_summary.sort_values(
    "total_reorder_point_units",
    ascending=False
)

### Store-level reorder-point interpretation

Each store contains 3,049 product series. Summing the per-product reorder-point estimates gives CA_3 the highest combined threshold (about 64,720 units) and CA_4 the lowest (about 32,126 units).

These are scenario estimates based on the assumed lead time and service level. They describe estimated stock thresholds across products, not current inventory or quantities to order.

### Assumptions and limitations

- The forecast uses the 7-day average baseline to estimate demand for the next 28 days.
- The reorder-point example assumes a 7-day supplier lead time and a 95% service level.
- Safety stock is estimated from sales variability over the most recent 28 days.
- The dataset does not include actual stock on hand, incoming orders, or supplier lead times. Therefore, these reorder points are educational estimates, not real Walmart replenishment instructions.
- The model uses historical sales and does not account for promotions, price changes, or future events.

In [ ]:
processed_dir = project_dir / "data" / "processed"

output_files = [
    processed_dir / "forecast_7day_average_28day.csv",
    processed_dir / "reorder_point_scenario_7day_average.csv",
]

for output_file in output_files:
    print(
        output_file.name,
        "— saved" if output_file.exists() else "— not found"
    )

### Simulated supply inputs

M5 provides the sales history and forecast, but not stock or supplier records. For this scenario, I will generate starting stock equal to a randomly selected 0–14 days of forecast demand for each product–store series. I will use a fixed random seed so the scenario can be reproduced.

I will assume there are no incoming orders, a 7-day supplier lead time, and a 95% service level. These supply inputs are simulated for demonstration; they are not Walmart records.

In [ ]:
supply_rng = np.random.default_rng(50)

simulated_supply_inputs = forecast_with_metadata[
    [
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
        "forecast_28day_total",
    ]
].copy()

simulated_supply_inputs["simulated_stock_cover_days"] = supply_rng.integers(
    low=0,
    high=15,  # includes 0 through 14 days
    size=len(simulated_supply_inputs),
)

simulated_supply_inputs["simulated_on_hand_units"] = np.ceil(
    simulated_supply_inputs["forecast_28day_total"]
    / 28
    * simulated_supply_inputs["simulated_stock_cover_days"]
).astype(int)

simulated_supply_inputs["simulated_on_order_units"] = 0
simulated_supply_inputs["assumed_lead_time_days"] = 7

simulated_supply_inputs.shape, simulated_supply_inputs.head()

In [ ]:
inventory_scenario = inventory_reorder_thresholds.merge(
    simulated_supply_inputs[
        [
            "id",
            "forecast_28day_total",
            "simulated_on_hand_units",
            "simulated_on_order_units",
            "assumed_lead_time_days",
        ]
    ],
    on="id",
    how="left",
    validate="one_to_one",
)

inventory_scenario["simulated_inventory_position_units"] = (
    inventory_scenario["simulated_on_hand_units"]
    + inventory_scenario["simulated_on_order_units"]
)

inventory_scenario["simulated_reorder_now"] = (
    inventory_scenario["simulated_inventory_position_units"]
    <= inventory_scenario["reorder_point_units_rounded"]
)

inventory_scenario[
    [
        "id",
        "forecast_28day_total",
        "simulated_on_hand_units",
        "reorder_point_units_rounded",
        "simulated_inventory_position_units",
        "simulated_reorder_now",
    ]
].head(10)

In [ ]:
reorder_count = inventory_scenario["simulated_reorder_now"].sum()
series_count = len(inventory_scenario)
reorder_percent = 100 * reorder_count / series_count

print(f"Flagged for reorder: {reorder_count:,} of {series_count:,} series")
print(f"Percentage flagged: {reorder_percent:.1f}%")

In [ ]:
reorder_by_store = (
    inventory_scenario.groupby("store_id")
    .agg(
        series_count=("id", "count"),
        flagged_for_reorder=("simulated_reorder_now", "sum"),
    )
)

reorder_by_store["flagged_percent"] = (
    100 * reorder_by_store["flagged_for_reorder"]
    / reorder_by_store["series_count"]
)

reorder_by_store.sort_values("flagged_percent", ascending=False)

In [ ]:
inventory_scenario["simulated_order_up_to_level_units"] = np.ceil(
    inventory_scenario["expected_demand_during_lead_time"]
    + inventory_scenario["forecast_28day_total"]
    + inventory_scenario["safety_stock_units"]
).astype(int)

inventory_scenario["simulated_order_quantity_units"] = np.where(
    inventory_scenario["simulated_reorder_now"],
    np.maximum(
        inventory_scenario["simulated_order_up_to_level_units"]
        - inventory_scenario["simulated_inventory_position_units"],
        0,
    ),
    0,
).astype(int)

inventory_scenario.loc[
    inventory_scenario["simulated_reorder_now"],
    [
        "id",
        "simulated_inventory_position_units",
        "simulated_order_up_to_level_units",
        "simulated_order_quantity_units",
    ],
].head(10)

In [ ]:
simulated_order_summary_by_store = (
    inventory_scenario.groupby("store_id")
    .agg(
        series_count=("id", "count"),
        flagged_for_reorder=("simulated_reorder_now", "sum"),
        total_simulated_order_units=("simulated_order_quantity_units", "sum"),
    )
)

simulated_order_summary_by_store["flagged_percent"] = (
    100
    * simulated_order_summary_by_store["flagged_for_reorder"]
    / simulated_order_summary_by_store["series_count"]
)

simulated_order_summary_by_store.sort_values(
    "total_simulated_order_units",
    ascending=False
)

In [ ]:
plot_data = simulated_order_summary_by_store.sort_values(
    "total_simulated_order_units"
)

plt.figure(figsize=(9, 5))
plt.bar(
    plot_data.index,
    plot_data["total_simulated_order_units"],
)
plt.title("Simulated Order Quantity by Store")
plt.xlabel("Store")
plt.ylabel("Simulated order units")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
processed_dir.mkdir(parents=True, exist_ok=True)

scenario_file = (
    processed_dir / "simulated_inventory_replenishment_scenario.csv"
)
store_summary_file = (
    processed_dir / "simulated_store_order_summary.csv"
)

inventory_scenario.to_csv(scenario_file, index=False)

simulated_order_summary_by_store.to_csv(
    store_summary_file,
    index=True,
    index_label="store_id",
)

print("Saved:", scenario_file.name)
print("Saved:", store_summary_file.name)

### Simulated supply scenario results

Under the generated starting-stock assumptions, 27,578 of 30,490 product–store series (90.4%) were flagged for reorder. CA_3 had the largest total simulated order quantity (143,073 units), while CA_4 had the smallest (75,143 units).

These results depend on the simulated stock levels and assumed replenishment policy. They are not Walmart’s actual stock or order requirements. The detailed scenario and store summary are saved in `data/processed/`.

## Simulated inventory scenario

M5 does not include actual stock or supplier records. To demonstrate replenishment planning, this project generates simulated starting stock for each product–store series and assumes no incoming orders, a 7-day lead time, and a 95% service level.

In this scenario, 27,578 of 30,490 series (90.4%) were flagged for reorder. These are illustrative results, not Walmart’s actual inventory or order requirements.

The processed folder contains the detailed simulated scenario and a store-level summary.